# Infra-Bench -- OlmoEarth v1.1-Base Full Fine-Tune, 0.3x training data

**Full fine-tune of OlmoEarth v1.1-Base on a stratified 30% subsample of the training set.** Val and test sets are unchanged, so test macro F1 is directly comparable to the 1.0x variant.

**Full fine-tuning evaluation of OlmoEarth v1.1-Base.** Complements the
linear-probe result reported in `infra_fm_olmoearth_lp_v1.ipynb`.

## What's different vs. the LP notebook

| Aspect | Linear probe | Full fine-tune |
|---|---|---|
| Backbone | Frozen (`freeze=True`) | Fully trainable (`freeze=False`) |
| Backbone LR | 1e-3 (uniform on head) | **6e-5** base x LLRD gamma=0.75 per depth |
| Head LR | 1e-3 | **1e-3** (separate param group) |
| Weight decay | 1e-4 | **0.05** (Prithvi FT recipe) |
| LR schedule | None | **Cosine + 5% linear warmup**, per-effective-step stepping |
| Epochs | 25 | 25 (unchanged) |
| Batch size | 16 | 16 nominal; adaptive fallback if OOM |
| Autocast | none | **fp16 autocast + GradScaler** (train only) |
| Grad checkpointing | none | conditional (see MEMORY_FALLBACK_LEVEL) |

## Backbone class

Reuses the LP-working `OlmoEarthBackbone`: encoder-only invocation
(`self.backbone.encoder(sample, patch_size=8)`), `MaskedOlmoEarthSample` with
an all-False `sentinel2_l2a_mask`, `project_aggregated` pooling head. The
only functional change is `freeze=False` so the entire model (including the
projection aggregator) is trainable.

## Outputs (only after `SMOKE_ONLY=False`)

- `.../results/fm_eval_olmoearth_finetune_0.3x_v1/olmoearth_finetune_0.3x_v1_seed{{314,271,161}}_results.json`
- `.../results/fm_eval_olmoearth_finetune_0.3x_v1/olmoearth_finetune_0.3x_v1_aggregate.json`
- `.../results/fm_eval_olmoearth_finetune_0.3x_v1/confusion_matrix_olmoearth_finetune_0.3x_v1_aggregate.png`

## Smoke test

Default `SMOKE_ONLY=True` runs 2 epochs on seed 314 with the full 25-epoch
LR schedule previewed. Flip to False to launch the 3-seed run.

## Aggregate guard

Aggregate JSON write is gated by `set(SEEDS) == set(FULL_PROTOCOL_SEEDS)`.
Partial (1- or 2-seed) reruns will not overwrite an existing 3-seed aggregate.


In [1]:
# Colab's base image already has a coherent torch/torchvision/torchaudio triple;
# force-reinstalling torch breaks torchaudio ABI. olmoearth_pretrain_minimal
# requires torch>=2.7 which Colab's default already satisfies. Don't touch
# torch/torchvision/numpy -- let Colab's base image provide the CUDA+ABI-
# consistent tuple.
!pip install -q olmoearth_pretrain_minimal
!pip install -q pyarrow scikit-learn scipy huggingface-hub==0.36.2

# Verify torch/torchvision loaded cleanly against the current CUDA
!python -c "import torch, torchvision; print(f'torch={torch.__version__}, torchvision={torchvision.__version__}, cuda_ok={torch.cuda.is_available()}')"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.12.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch=2.11.0+cu128, torchvision=0.26.0+cu128, cuda_ok=True


In [2]:
# Pre-flight: check what's actually on disk for the critical libraries.
# Pip queries the filesystem, not Python's in-memory imports, so this
# tells us the truth regardless of what's loaded.
import subprocess

critical = ['torch', 'torchvision', 'pillow', 'numpy', 'scipy',
            'scikit-learn', 'olmoearth_pretrain_minimal',
            'huggingface-hub', 'pyarrow']

for pkg in critical:
    result = subprocess.run(
        ['pip', 'show', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                print(f'  {pkg:<20s} {line.split(":", 1)[1].strip()}')
                break
    else:
        print(f'  {pkg:<20s} NOT INSTALLED')


  torch                2.11.0+cu128
  torchvision          0.26.0+cu128
  pillow               11.3.0
  numpy                2.0.2
  scipy                1.16.3
  scikit-learn         1.6.1
  olmoearth_pretrain_minimal 0.0.6
  huggingface-hub      0.36.2
  pyarrow              18.1.0


In [3]:
# Conditional numpy restore. Some Colab images resolve pip installs
# in a way that pins numpy down to 1.x, which then breaks sklearn 1.5+.
# Only force the upgrade if we detect the downgrade. When we do upgrade,
# raise to force a clean kernel restart -- pip cannot swap numpy in an
# already-loaded kernel.
import numpy as np
if np.__version__.startswith('1.'):
    print(f'numpy is on {np.__version__} (1.x). Restoring to 2.x and forcing a restart...')
    !pip install --upgrade --force-reinstall "numpy>=2.2"
    !pip install --upgrade --force-reinstall --no-deps scikit-learn scipy
    raise RuntimeError(
        'numpy was pinned to 1.x by the install above; restored to 2.x. '
        'Restart the Colab kernel (Runtime -> Restart session) and re-run '
        'from the top.'
    )
else:
    print(f'numpy {np.__version__} OK (no restore needed)')

import sklearn, scipy
print(f'sklearn  {sklearn.__version__}')
print(f'scipy    {scipy.__version__}')


numpy 2.0.2 OK (no restore needed)
sklearn  1.6.1
scipy    1.16.3


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


Mounted at /content/drive
GPU available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [5]:
import torch
print(f"torch: {torch.__version__}")
print(f"torchvision available:", end=" ")
try:
    import torchvision
    print(f"{torchvision.__version__}")
except Exception as e:
    print(f"FAILED -- {e}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
try:
    from torchvision.ops import nms
    print("torchvision.ops.nms: OK")
except Exception as e:
    print(f"torchvision.ops.nms: FAILED -- {e}")


torch: 2.11.0+cu128
torchvision available: 0.26.0+cu128
CUDA available: True
CUDA version: 12.8
torchvision.ops.nms: OK


In [6]:
# ============================================================================
# HF authentication -- resilient chain: Colab Secrets -> .env on Drive -> env
# ----------------------------------------------------------------------------
# The token is only a fallback for HF metadata calls. Primary weight load
# path is the Drive HF cache with HF_HUB_OFFLINE=1, so "no token found"
# is not a failure in this notebook.
# ============================================================================
import os

hf_token = None

# 1. Colab Secrets (works when notebook is running interactively in the UI)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        print('  HF_TOKEN found via Colab Secrets')
except Exception:
    pass  # userdata unavailable (background exec, timeout, non-Colab, etc.)

# 2. .env on Drive (portable across sessions; survives runtime restarts)
if not hf_token:
    try:
        from dotenv import load_dotenv
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'python-dotenv'], check=False)
        from dotenv import load_dotenv
    for env_path in ['/content/drive/MyDrive/infra_fm/.env',
                     '/content/drive/MyDrive/.env']:
        if os.path.exists(env_path):
            load_dotenv(env_path)
            hf_token = os.environ.get('HF_TOKEN')
            if hf_token:
                print(f'  HF_TOKEN found via {env_path}')
                break

# 3. Any HF_TOKEN already in os.environ (e.g. set by an earlier cell)
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  HF_TOKEN found via os.environ')

# 4. Login if we got a token; otherwise proceed anonymously (Drive cache carries us)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN via userdata / .env / environ -- proceeding anonymously.')
    print('(Fine when HF_HUB_OFFLINE=1 and models are pre-loaded in Drive cache.)')


  HF_TOKEN found via /content/drive/MyDrive/infra_fm/.env


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace.


In [7]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Only load_split_artifact is needed at training time. If the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import (zip is pre-Phase-1): No module named 'curation.utils.spatial_blocking'. Using inline fallback.


In [8]:
\
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_olmoearth_finetune_0.3x_v1'
SPLIT_ARTIFACT_PATH = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.water_works',   # legacy manifest tag
    'water.water_works':                      'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# OlmoEarth band remap. Our .npy storage:
#   [B04, B03, B02, B08, B8A, B11, B12, VV, VH]   indices 0..8
# We select the same 6 bands as Prithvi (order [B02, B03, B04, B8A, B11, B12])
# so the on-disk layout and percentile-normalize path are shared.
OLMOEARTH_BAND_INDICES = [2, 1, 0, 4, 5, 6]

PERC_LO, PERC_HI = 2.0, 98.0
IMAGE_SIZE = 224
WEIGHT_CAP = 10.0
SEEDS      = [314, 271, 161]

# ---- Fine-tune hyperparameters (borrowed from Prithvi FT recipe verbatim) ----
FT_EPOCHS                  = 25
FT_BACKBONE_LR             = 6e-5        # backbone base LR (LLRD applies per depth)
FT_HEAD_LR                 = 1e-3        # head LR (separate param group)
FT_WD                      = 0.05
FT_LLRD_GAMMA              = 0.75        # LR x gamma^(max_depth - depth)
FT_WARMUP_FRACTION         = 0.05        # 5% of total effective steps

RESUME_FROM_CHECKPOINT      = True
AUTO_SKIP_SMOKE_IF_RESUMING = True

RUN_NAME_PREFIX            = 'olmoearth_finetune_0.3x_v1'

# ---- Aggregate config -----------------------------------------------------
# Full-protocol seeds. Aggregate JSON write is gated on
# set(SEEDS) == set(FULL_PROTOCOL_SEEDS) so partial reruns don't overwrite
# a 3-seed aggregate.
FULL_PROTOCOL_SEEDS = [314, 271, 161]

# ---- Memory fallback control (mirrors Prithvi FT) -------------------------
MEMORY_FALLBACK_LEVEL = 0

if MEMORY_FALLBACK_LEVEL == 0:
    FT_BATCH               = 16
    FT_GRAD_ACCUM_STEPS    = 1
    FT_GRAD_CHECKPOINTING  = False
elif MEMORY_FALLBACK_LEVEL == 1:
    FT_BATCH               = 16
    FT_GRAD_ACCUM_STEPS    = 1
    FT_GRAD_CHECKPOINTING  = True
elif MEMORY_FALLBACK_LEVEL == 2:
    FT_BATCH               = 8
    FT_GRAD_ACCUM_STEPS    = 2
    FT_GRAD_CHECKPOINTING  = True
else:
    raise ValueError(f'MEMORY_FALLBACK_LEVEL must be 0, 1, or 2; got {MEMORY_FALLBACK_LEVEL}')

FT_EFFECTIVE_BATCH = FT_BATCH * FT_GRAD_ACCUM_STEPS
assert FT_EFFECTIVE_BATCH == 16, (
    f'Effective batch drifted from 16 ({FT_EFFECTIVE_BATCH}); '
    'check MEMORY_FALLBACK_LEVEL settings.'
)

FINETUNE_PROTOCOL = {
    'backbone_lr':          FT_BACKBONE_LR,       # base -- actual per-depth LR varies via LLRD
    'head_lr':              FT_HEAD_LR,
    'weight_decay':         FT_WD,
    'scheduler':            'cosine_with_linear_warmup',
    'warmup_fraction':      FT_WARMUP_FRACTION,
    'llrd':                 FT_LLRD_GAMMA,
    'autocast':             True,
    'grad_checkpointing':   FT_GRAD_CHECKPOINTING,
    'grad_accum_steps':     FT_GRAD_ACCUM_STEPS,
    'epochs':               FT_EPOCHS,
    'batch_size':           FT_BATCH,
    'effective_batch_size': FT_EFFECTIVE_BATCH,
    'memory_fallback_level': MEMORY_FALLBACK_LEVEL,
    'class_weight_cap':     WEIGHT_CAP,
    'seeds':                list(SEEDS),
    'unfreeze_scope':       'all_backbone_params_LLRD_gamma_0.75',
    'runtime_gpu':          'A100 (Colab)',
}

# ---- 0.3x stratified subsampling of TRAINING set (val + test unchanged) ----
# SUBSAMPLE_SEED is fixed across the 3 training seeds so seed-variance and
# subsample-variance are separated cleanly.
SUBSAMPLE_FRACTION = 0.3
SUBSAMPLE_SEED     = 42


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:            {OUTPUT_DIR}')
print(f'Split artifact:        {SPLIT_ARTIFACT_PATH}')
print(f'OlmoEarth band indices: {OLMOEARTH_BAND_INDICES}  (B02, B03, B04, B8A, B11, B12)')
print(f'Training seeds:        {SEEDS}')
print(f'Fine-tune protocol:    {FT_EPOCHS} epochs, effective batch {FT_EFFECTIVE_BATCH}')
print(f'  backbone_base_lr:    {FT_BACKBONE_LR}   (LLRD gamma={FT_LLRD_GAMMA})')
print(f'  head_lr:             {FT_HEAD_LR}   (separate param group)')
print(f'  weight_decay:        {FT_WD}')
print(f'  scheduler:           cosine + {FT_WARMUP_FRACTION*100:.0f}% warmup')
print(f'  autocast:            fp16 (train only; eval stays fp32)')
print(f'MEMORY_FALLBACK_LEVEL: {MEMORY_FALLBACK_LEVEL}')
print(f'  batch_size:          {FT_BATCH}')
print(f'  grad_accum_steps:    {FT_GRAD_ACCUM_STEPS}')
print(f'  grad_checkpointing:  {FT_GRAD_CHECKPOINTING}')


Output dir:            /content/drive/MyDrive/infra_fm/results/fm_eval_olmoearth_finetune_0.3x_v1
Split artifact:        /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
OlmoEarth band indices: [2, 1, 0, 4, 5, 6]  (B02, B03, B04, B8A, B11, B12)
Training seeds:        [314, 271, 161]
Fine-tune protocol:    25 epochs, effective batch 16
  backbone_base_lr:    6e-05   (LLRD gamma=0.75)
  head_lr:             0.001   (separate param group)
  weight_decay:        0.05
  scheduler:           cosine + 5% warmup
  autocast:            fp16 (train only; eval stays fp32)
MEMORY_FALLBACK_LEVEL: 0
  batch_size:          16
  grad_accum_steps:    1
  grad_checkpointing:  False


In [9]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


  [EXTRACT]africa                 energy     4s (949 tiles)
  [EXTRACT]africa                 telecom    2s (1 tiles)
  [EXTRACT]africa                 transport  5s (1000 tiles)
  [EXTRACT]africa                 water      5s (892 tiles)
  [EXTRACT]asia                   energy     4s (889 tiles)
  [EXTRACT]asia                   telecom    3s (28 tiles)
  [EXTRACT]asia                   transport  7s (891 tiles)
  [EXTRACT]asia                   water      5s (958 tiles)
  [EXTRACT]australia-oceania      energy     5s (1002 tiles)
  [EXTRACT]australia-oceania      telecom    3s (12 tiles)
  [EXTRACT]australia-oceania      transport  5s (1000 tiles)
  [EXTRACT]australia-oceania      water      6s (1000 tiles)
  [EXTRACT]central-america        energy     5s (999 tiles)
  [EXTRACT]central-america        telecom    2s (1 tiles)
  [EXTRACT]central-america        transport  5s (566 tiles)
  [EXTRACT]central-america        water      6s (999 tiles)
  [EXTRACT]europe                 energy  

In [10]:
\
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class OlmoEarthDataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects the 6 shared S2 bands per `OLMOEARTH_BAND_INDICES`, applies
    percentile_normalize, and unsqueezes a T=1 temporal dim so the default
    collate stacks to `(B, 6, 1, H, W)`."""
    def __init__(self, dataset_root,
                 band_indices=OLMOEARTH_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'

        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed:
                dropped['filtered_type' if at else 'no_label'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  OlmoEarthDataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (6, H, W) in the shared S2 band order
        arr = percentile_normalize(arr)                  # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)                   # (6, H, W)
        t = torch.from_numpy(img).unsqueeze(1)           # (6, 1, H, W)  -- T=1 dim
        return {'image': t, 'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']                            # (6, 1, H, W)
        C, T, H, W = img.shape
        img = img.reshape(C * T, 1, H, W)
        img = F.interpolate(img, size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False)
        img = img.reshape(C, T, self.input_size, self.input_size)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = OlmoEarthDataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


Built 28 cell datasets


In [11]:
# Load the spatial split artifact and slice each cell into train/val/test.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles have no split assignment (excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])


# ---- Stratified 0.3x subsampling of TRAINING set (val + test unchanged) ----
# Preserves per-class proportions. Deterministic via SUBSAMPLE_SEED so all
# 3 training seeds see the same 30% subsample.
import random as _random
from collections import defaultdict as _defaultdict


class _StratifiedSubsample(Dataset):
    """Wraps a ConcatDataset with a subset of indices. Exposes `.labels`
    so the training-infra cell's `_labels_from(dataset)` (defined later)
    hits its `else` branch and reads labels cleanly."""
    def __init__(self, base, indices, labels):
        self.base = base
        self.indices = list(indices)
        self.labels = list(labels)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


def _iter_concat_labels(concat_ds):
    global_idx = 0
    for sub in concat_ds.datasets:
        if not isinstance(sub, SubsetView):
            raise ValueError(f'Unexpected sub-dataset type: {type(sub)}')
        for local_i in range(len(sub)):
            yield global_idx, sub.base.labels[sub.indices[local_i]]
            global_idx += 1


def _apply_stratified_subsample(dataset, fraction, seed):
    idx_labels = list(_iter_concat_labels(dataset))
    by_class = _defaultdict(list)
    for idx, label in idx_labels:
        by_class[label].append(idx)
    rng = _random.Random(seed)
    selected = []
    for label in sorted(by_class):
        class_idxs = by_class[label].copy()
        rng.shuffle(class_idxs)
        n_take = max(1, int(round(len(class_idxs) * fraction)))
        chosen = class_idxs[:n_take]
        selected.extend((i, label) for i in chosen)
    selected.sort()
    return _StratifiedSubsample(dataset,
                                 indices=[i for i, _ in selected],
                                 labels=[l for _, l in selected])


_orig_train_size = len(train_global)
train_global = _apply_stratified_subsample(
    train_global, SUBSAMPLE_FRACTION, SUBSAMPLE_SEED,
)
print(f'\nStratified {SUBSAMPLE_FRACTION:.1%} subsample of TRAINING set:')
print(f'  before: {_orig_train_size:>6d} tiles')
print(f'  after:  {len(train_global):>6d} tiles   '
      f'(seed={SUBSAMPLE_SEED}; val + test unchanged)')

from collections import Counter as _Counter
_class_counts = _Counter(train_global.labels)
print('  per-class counts (subsampled train):')
for _c in sorted(_class_counts):
    _name = CLASS_NAMES[_c] if _c < len(CLASS_NAMES) else f'class_{_c}'
    print(f'    [{_c:>2d}] {_name:<34s} {_class_counts[_c]:>4d}')

print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


Loaded split artifact: 18,750 asset_id -> split entries
  splits distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})

Stratified 30.0% subsample of TRAINING set:
  before:  13087 tiles
  after:    3927 tiles   (seed=42; val + test unchanged)
  per-class counts (subsampled train):
    [ 0] energy.transmission.substation      138
    [ 1] energy.distribution.substation      201
    [ 2] energy.distribution.other           694
    [ 3] energy.generation.power_plant       110
    [ 4] energy.generation.solar_farm        130
    [ 5] energy.generation.wind_farm           2
    [ 6] water.wastewater.plant              251
    [ 7] water.water_works                   198
    [ 8] water.storage_tank                  840
    [ 9] transport.airport                   177
    [10] transport.train_station            1068
    [11] transport.port_terminal               2
    [12] telecom.data_center                 116

Global: train=3927  val=2856  test=2813


In [12]:
# Diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison vs the new spatial split.
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed -- same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)')


Diagnostic: train/val/test transition table (old random -> new spatial)
Comparing on 18,750 tiles in both old and new splits

old \ new      train       val       test
--------------------------------------------------
train           9120      1983       1978
val             1946       410        415
test            2021       458        419

Unchanged: 9,949 (53.1%)
Changed:   8,801 (46.9%)

(Expected ~47% changed -- same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)


In [13]:
import torch.nn as nn
from olmoearth_pretrain_minimal import load_model_from_id, ModelID, Normalizer
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.constants import Modality
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample


class OlmoEarthBackbone(nn.Module):
    """OlmoEarth v1.1-Base backbone. For FT we set freeze=False so the entire
    model (including the projection head) is trainable; the LP-working
    encoder-only invocation is preserved."""
    NAME = 'olmoearth_v1_1_base'
    HF_REPO = 'allenai/OlmoEarth-v1-Base'
    EXPECTED_FEATURE_DIM = 768  # ViT-Base

    def __init__(self, freeze=False):
        super().__init__()
        self.normalizer = Normalizer(std_multiplier=2.0)
        try:
            self.backbone = load_model_from_id(ModelID.OLMOEARTH_V1_1_BASE, load_weights=True)
            self._version = 'v1.1-Base'
        except AttributeError:
            self.backbone = load_model_from_id(ModelID.OLMOEARTH_V1_BASE, load_weights=True)
            self._version = 'v1-Base (fallback)'
            print('  WARNING: v1.1-Base not exposed by olmoearth_pretrain_minimal; using v1-Base')
        # For FT we keep the module in train mode; freeze=False is the default.
        if freeze:
            self.backbone.eval()
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.feature_dim = self._infer_feature_dim()

    def _prepare_input(self, x):
        """Convert Infra-Bench (B, 6, 1, H, W) to OlmoEarth's expected
        (B, H, W, T=1, 12) with the full 12-band S2 order.
        Our 6 stored bands: [B02, B03, B04, B8A, B11, B12] (Prithvi order after
        OLMOEARTH_BAND_INDICES selection). We pad B01, B05, B06, B07, B08, B09
        with zeros. Relies on v1.1's band-dropout robustness."""
        B, C, T, H, W = x.shape
        assert C == 6 and T == 1, f'expected (B, 6, 1, H, W), got {x.shape}'
        twelve_band = torch.zeros(B, 12, T, H, W, device=x.device, dtype=x.dtype)
        # OlmoEarth band order: [B02, B03, B04, B08, B05, B06, B07, B8A, B11, B12, B01, B09]
        # Our indices (Prithvi):   B02=0, B03=1, B04=2, B8A=3, B11=4, B12=5
        twelve_band[:, 0] = x[:, 0]   # B02
        twelve_band[:, 1] = x[:, 1]   # B03
        twelve_band[:, 2] = x[:, 2]   # B04
        # index 3 (B08) stays zero -- Prithvi uses B8A not B08
        # indices 4, 5, 6 (B05, B06, B07) stay zero -- band dropout
        twelve_band[:, 7] = x[:, 3]   # B8A
        twelve_band[:, 8] = x[:, 4]   # B11
        twelve_band[:, 9] = x[:, 5]   # B12
        # indices 10, 11 (B01, B09) stay zero -- band dropout
        # Permute FIRST -- moves bands from dim 1 to last dim: (B, H, W, T, 12)
        twelve_band = twelve_band.permute(0, 3, 4, 2, 1).contiguous()
        
        # THEN scale (after permute -- order doesn't affect scaling, but keep it visually near normalize)
        twelve_band = twelve_band * 10000.0
        
        # THEN normalize
        arr = twelve_band.detach().cpu().numpy()
        arr = self.normalizer.normalize(Modality.SENTINEL2_L2A, arr)
        return torch.from_numpy(arr).to(device=x.device, dtype=torch.float32)

    def _build_sample(self, s2_tensor):
        """Wrap normalized S2 tensor in MaskedOlmoEarthSample with required timestamps + masks."""
        B, H, W, T, D = s2_tensor.shape   # (B, H, W, T, 12)

        # Timestamps: (B, T, 3) with [year_offset, month_0_to_11, day_offset]
        timestamps = torch.zeros(B, T, 3, dtype=torch.long, device=s2_tensor.device)
        timestamps[:, :, 1] = 6  # month = June

        # Modality mask: pixel-resolution bool tensor. False = visible token.
        # For feature extraction we want no masking: all tokens visible.
        s2_mask = torch.zeros(B, H, W, T, dtype=torch.bool, device=s2_tensor.device)

        sample = MaskedOlmoEarthSample(
            timestamps=timestamps,
            sentinel2_l2a=s2_tensor,
            sentinel2_l2a_mask=s2_mask,
        )
        return sample

    def _infer_feature_dim(self):
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 6, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        was_training = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            feat = self.forward(dummy)
        if was_training:
            self.backbone.train()
        d = feat.shape[-1]
        print(f'  feature_dim = {d}')
        return d

    def forward(self, x):
        prepared = self._prepare_input(x)
        sample = self._build_sample(prepared)
        # Encoder-only invocation (skips the decoder used for pretraining
        # reconstruction). patch_size mirrors the LP-working path.
        out = self.backbone.encoder(sample, patch_size=8)
        return self._extract_features(out)

    @staticmethod
    def _extract_features(out):
        """Extract a pooled (B, D) feature vector from OlmoEarth's encoder output dict.

        OlmoEarth v1.1-Base encoder returns:
        - 'project_aggregated': (B, 768) -- the pooled global feature. THIS is what we want.
        - 'tokens_and_masks': TokensAndMasks object holding per-token features + masks.
        """
        if isinstance(out, dict) and 'project_aggregated' in out:
            return out['project_aggregated']
        if isinstance(out, dict):
            for key in ('latent_projected_and_pooled', 'pooled', 'cls', 'global_feat'):
                if key in out and isinstance(out[key], torch.Tensor) and out[key].dim() == 2:
                    return out[key]
            raise RuntimeError(f'Cannot extract feature from encoder dict: keys={list(out.keys())}')
        if hasattr(out, 'shape'):
            if out.dim() == 2:
                return out
            if out.dim() == 3:
                return out.mean(dim=1)
            if out.dim() == 4:
                return out.mean(dim=[1, 2])
        raise RuntimeError(f'Unexpected encoder output: {type(out).__name__}')


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


def BACKBONE_FACTORY(freeze=False):
    return OlmoEarthBackbone(freeze=freeze)


print('OlmoEarthBackbone + InfraBenchClassifier defined (freeze default = False for FT).')


OlmoEarthBackbone + InfraBenchClassifier defined (freeze default = False for FT).


In [14]:
import re as _re
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """v2 per-sector -- matches CROMA v2 schema exactly.
    Mean of per-class F1s for the classes in that sector, computed on the
    FULL test set."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    """Eval stays in fp32 to preserve F1 metric fidelity -- autocast is training-only."""
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition).'
        )
    return result


# ============================================================================
# LLRD (layer-wise LR decay) param-group builder
# ----------------------------------------------------------------------------
# Discovers block indices from the backbone's parameter names using a
# broadened regex that matches both `blocks.N.` (Prithvi / OlmoEarth) and
# `layer.N.` (HF Transformers -- DINOv3). Non-block params (patch_embed,
# cls_token, pos_embed, pre-block norm) go into depth 0 (lowest LR).
# If no block pattern is found we fall back to a single "all params in
# depth 0" group (still trainable at base_lr).
# ============================================================================
def build_llrd_param_groups(backbone, head, base_lr, head_lr, gamma, wd,
                             verbose=True):
    block_pat = _re.compile(r'(?:blocks?|layer|layers)\.(\d+)\.')
    block_indices = set()
    for name, _ in backbone.named_parameters():
        m = block_pat.search(name)
        if m:
            block_indices.add(int(m.group(1)))

    if not block_indices:
        if verbose:
            print('  LLRD: WARNING -- no block pattern matched. Falling back to a '
                  'single backbone param group at base_lr.')
        bb_params = [p for p in backbone.parameters() if p.requires_grad]
        groups = [
            {'params': bb_params, 'lr': base_lr, 'weight_decay': wd, 'name': 'bb_all'},
        ]
        head_params = [p for p in head.parameters() if p.requires_grad]
        groups.append({'params': head_params, 'lr': head_lr, 'weight_decay': wd, 'name': 'head'})
        # Diagnostics: single depth 0 entry per param.
        diagnostics = [(name, 0, base_lr) for name, p in backbone.named_parameters()
                       if p.requires_grad]
        return groups, diagnostics

    max_block = max(block_indices)
    max_depth = max_block + 1   # +1 for the patch_embed / pre-block layers at depth 0

    per_depth = defaultdict(list)
    diagnostics = []
    for name, p in backbone.named_parameters():
        if not p.requires_grad:
            continue
        m = block_pat.search(name)
        depth = int(m.group(1)) + 1 if m else 0
        lr = base_lr * (gamma ** (max_depth - depth))
        per_depth[depth].append((name, p, lr))
        diagnostics.append((name, depth, lr))

    groups = []
    for depth in sorted(per_depth.keys()):
        entries = per_depth[depth]
        lr = entries[0][2]
        groups.append({
            'params':       [p for (_, p, _) in entries],
            'lr':           lr,
            'weight_decay': wd,
            'name':         f'bb_depth{depth}',
        })
    head_params = [p for p in head.parameters() if p.requires_grad]
    groups.append({
        'params':       head_params,
        'lr':           head_lr,
        'weight_decay': wd,
        'name':         'head',
    })

    if verbose:
        print(f'  LLRD: {len(block_indices)} blocks, max_depth={max_depth}, '
              f'gamma={gamma}, base_lr={base_lr}')
        print(f'  LLRD: {len(groups)} param groups (backbone: {len(groups)-1} x depth-buckets, '
              f'+ 1 head group at lr={head_lr})')
        depth_lrs = sorted(set((d, per_depth[d][0][2]) for d in per_depth), key=lambda x: x[0])
        print(f'  LLRD depth->lr: shallowest={depth_lrs[0][1]:.2e} (depth 0), '
              f'deepest={depth_lrs[-1][1]:.2e} (depth {max_depth})')

    return groups, diagnostics


def train_one_seed(seed, *, train_set, val_set, test_set,
                   num_epochs=None, scheduler_total_epochs=None,
                   allow_resume=None, run_name_override=None):
    """Full fine-tune training loop: LLRD + autocast + optional grad accum
    + optional grad checkpointing. Mirrors Prithvi FT structurally."""
    if num_epochs is None:
        num_epochs = FT_EPOCHS
    if scheduler_total_epochs is None:
        scheduler_total_epochs = num_epochs
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    set_seed(seed)
    print(f'\n--- seed {seed}  fine-tune ---')
    print(f'    training loop: {num_epochs} epochs   scheduler length: {scheduler_total_epochs} epochs')
    print(f'    batch={FT_BATCH}  grad_accum={FT_GRAD_ACCUM_STEPS}  '
          f'effective_batch={FT_EFFECTIVE_BATCH}  grad_ckpt={FT_GRAD_CHECKPOINTING}')

    backbone = BACKBONE_FACTORY(freeze=False)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params:     {total_params:>13,}')
    print(f'  Trainable params: {trainable_params:>13,}')

    # Optional gradient checkpointing (best effort)
    grad_ckpt_active = False
    if FT_GRAD_CHECKPOINTING:
        target = backbone.backbone if hasattr(backbone, 'backbone') else backbone
        if hasattr(target, 'gradient_checkpointing_enable'):
            target.gradient_checkpointing_enable()
            grad_ckpt_active = True
            print('  gradient_checkpointing_enable() called on backbone')
        else:
            print('  WARNING: FT_GRAD_CHECKPOINTING=True but backbone lacks '
                  'gradient_checkpointing_enable(). Falling through with grad_ckpt OFF.')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # LLRD param groups
    param_groups, llrd_diag = build_llrd_param_groups(
        backbone=model.backbone, head=model.head,
        base_lr=FT_BACKBONE_LR, head_lr=FT_HEAD_LR,
        gamma=FT_LLRD_GAMMA, wd=FT_WD, verbose=True,
    )
    optimizer = AdamW(param_groups)

    # Cosine + warmup, per-*effective-step* stepping.
    steps_per_epoch_micro = len(train_loader)
    effective_steps_per_epoch = max(steps_per_epoch_micro // FT_GRAD_ACCUM_STEPS, 1)
    total_eff_steps = scheduler_total_epochs * effective_steps_per_epoch
    warmup_iters = max(1, int(FT_WARMUP_FRACTION * total_eff_steps))
    warmup_sched = LinearLR(optimizer, start_factor=1e-3, end_factor=1.0,
                             total_iters=warmup_iters)
    cosine_sched = CosineAnnealingLR(optimizer,
                                      T_max=max(total_eff_steps - warmup_iters, 1))
    scheduler    = SequentialLR(optimizer,
                                schedulers=[warmup_sched, cosine_sched],
                                milestones=[warmup_iters])

    scaler = GradScaler()

    run_name = run_name_override or f'{RUN_NAME_PREFIX}_seed{seed}_finetune'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt     = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt    = ckpt_dir / 'checkpoint_final.pt'
    metrics_jsonl = ckpt_dir / 'metrics.jsonl'

    # Resume-from-checkpoint: pick up from checkpoint_final.pt if present.
    history, best_val_f1, best_epoch = [], -1.0, -1
    start_epoch = 0
    resumed_epochs = 0
    if allow_resume and final_ckpt.exists():
        ckpt = torch.load(final_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            try:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            except Exception:
                pass
        if 'scaler_state_dict' in ckpt:
            try:
                scaler.load_state_dict(ckpt['scaler_state_dict'])
            except Exception:
                pass
        history = ckpt.get('history', [])
        best_val_f1 = ckpt.get('best_val_f1', -1.0)
        best_epoch = ckpt.get('best_epoch', -1)
        start_epoch = int(ckpt.get('epoch', 0))
        resumed_epochs = start_epoch
        # Rewrite metrics.jsonl from checkpoint history (authoritative).
        with metrics_jsonl.open('w', encoding='utf-8') as _mf:
            for _entry in history:
                _mf.write(json.dumps(_entry) + '\n')
            _mf.flush(); os.fsync(_mf.fileno())
        print(f'  [RESUME] Loaded {final_ckpt.name}: '
              f'{start_epoch}/{num_epochs} epochs done, '
              f'best_val_f1={best_val_f1:.4f} at epoch {best_epoch}')
    else:
        metrics_jsonl.write_text('', encoding='utf-8')

    t_seed_start = time.time()

    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        optimizer.zero_grad()
        for micro_step, batch in enumerate(train_loader):
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            with autocast(dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss / FT_GRAD_ACCUM_STEPS).backward()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
            if (micro_step + 1) % FT_GRAD_ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        current_lr = max(pg['lr'] for pg in optimizer.param_groups)
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'current_lr': current_lr,
            'time_s': time.time() - t0,
        }
        history.append(entry)
        with metrics_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry) + '\n'); f.flush(); os.fsync(f.fileno())

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)

        # Per-epoch checkpoint_final.pt for resume-safety on disconnect.
        _fckpt = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_f1': best_val_f1,
            'best_epoch': best_epoch,
        }
        try: _fckpt['scheduler_state_dict'] = scheduler.state_dict()
        except Exception: pass
        try: _fckpt['scaler_state_dict'] = scaler.state_dict()
        except Exception: pass
        torch.save(_fckpt, final_ckpt)

        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  lr_max={current_lr:.2e}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    # Persist final ckpt at end of training (idempotent w.r.t. per-epoch).
    torch.save({'epoch': num_epochs,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST ==================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)

    wall_time_s = time.time() - t_seed_start
    peak_gpu_gb = (torch.cuda.max_memory_allocated(DEVICE) / 1e9
                   if torch.cuda.is_available() else 0.0)

    # Cleanup: delete per-epoch resume checkpoints (best-ckpt already restored
    # + JSON is authoritative). Keeps Drive tidy across the 3-seed loop.
    for _ckpt in (final_ckpt, best_ckpt):
        if _ckpt.exists():
            try: _ckpt.unlink()
            except OSError: pass

    depth_lr_map = {}
    for name, depth, lr in llrd_diag:
        depth_lr_map.setdefault(depth, lr)

    return {
        'run_name':      run_name,
        'backbone':      backbone.NAME,
        'condition':     'full_finetune',
        'num_epochs':    num_epochs,
        'scheduler_total_epochs': scheduler_total_epochs,
        'seed':          seed,
        'best_val_f1':   best_val_f1,
        'best_epoch':    best_epoch,
        'tail_mean_f1':  float(np.mean(tail)),
        'tail_std_f1':   float(np.std(tail)),
        'history':       history,
        'resumed_epochs': int(resumed_epochs),
        'test':          test,
        'tested_with':   tested_with,
        'peak_gpu_gb':   float(peak_gpu_gb),
        'wall_time_s':   float(wall_time_s),
        'grad_ckpt_active': bool(grad_ckpt_active),
        'llrd_depth_lr_map': {int(d): float(lr) for d, lr in depth_lr_map.items()},
        'total_params':      int(total_params),
        'trainable_params':  int(trainable_params),
    }


print('Fine-tune training infrastructure ready.')
print('  - LLRD (gamma=%.2f) param groups + separate head group at head_lr=%.0e' % (FT_LLRD_GAMMA, FT_HEAD_LR))
print('  - AdamW, wd=%.2f' % FT_WD)
print('  - Cosine + %.0f%% linear warmup, per-effective-step stepping' % (FT_WARMUP_FRACTION * 100))
print('  - autocast(fp16) + GradScaler in training loop; eval stays fp32')
print(f'  - grad_accum={FT_GRAD_ACCUM_STEPS}, grad_ckpt={FT_GRAD_CHECKPOINTING}')


Device: cuda
Fine-tune training infrastructure ready.
  - LLRD (gamma=0.75) param groups + separate head group at head_lr=1e-03
  - AdamW, wd=0.05
  - Cosine + 5% linear warmup, per-effective-step stepping
  - autocast(fp16) + GradScaler in training loop; eval stays fp32
  - grad_accum=1, grad_ckpt=False


In [16]:
\
# ============================================================================
# SMOKE CHECK -- 2 epochs on seed 314.
# Prints the LLRD depth->LR mapping so you can eyeball it before committing
# to a full run. If the block-index regex didn't match the backbone's naming,
# you'll see nonsense values here (or a single "all params at depth 0" group
# from the LLRD fallback).
# ============================================================================
SMOKE_ONLY   = False
SMOKE_EPOCHS = 2
SMOKE_SEED   = SEEDS[0]

# Auto-skip smoke on resume: if a full-run checkpoint_final.pt exists for any
# seed, treat this as a reconnect and skip the 2-epoch smoke.
_full_run_ckpts_present = any(
    (Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{s}_finetune' / 'checkpoint_final.pt').exists()
    for s in SEEDS
)
if AUTO_SKIP_SMOKE_IF_RESUMING and _full_run_ckpts_present:
    print('=' * 76)
    print('AUTO_SKIP_SMOKE: existing full-run checkpoint_final.pt detected in one')
    print(f'or more of SEEDS={SEEDS} ckpt dirs. Interpreting this as a reconnect')
    print('after a disconnect -- skipping the 2-epoch smoke check.')
    print('=' * 76)
    SMOKE_ONLY = False
    smoke_result = None
else:
    print(f'SMOKE_ONLY = {SMOKE_ONLY}')
    print(f'Smoke check: {SMOKE_EPOCHS} epochs on seed {SMOKE_SEED}')
    print(f'MEMORY_FALLBACK_LEVEL = {MEMORY_FALLBACK_LEVEL}  '
          f'(bs={FT_BATCH}, accum={FT_GRAD_ACCUM_STEPS}, grad_ckpt={FT_GRAD_CHECKPOINTING})')
    print(f'Scheduler built for FULL {FT_EPOCHS}-epoch protocol (previews real LR trajectory).')
    print('=' * 76)

    print('\n[1] Instantiate FT model + inspect LLRD depth->LR mapping...')
    set_seed(SMOKE_SEED)
    sb = OlmoEarthBackbone(freeze=False)
    sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
    _total  = sum(p.numel() for p in sm.parameters())
    _train  = sum(p.numel() for p in sm.parameters() if p.requires_grad)
    print(f'  Total params:     {_total:>13,}')
    print(f'  Trainable params: {_train:>13,}   (expect >80,000,000 for full FT)')
    assert _train > 80000000, (
        f'Trainable param count {_train:,} looks too small for full fine-tune; '
        f'expected > 80,000,000. Backbone freeze may not have been disabled.'
    )

    smoke_groups, smoke_diag = build_llrd_param_groups(
        backbone=sm.backbone, head=sm.head,
        base_lr=FT_BACKBONE_LR, head_lr=FT_HEAD_LR,
        gamma=FT_LLRD_GAMMA, wd=FT_WD, verbose=True,
    )
    print('\n  LLRD depth->LR mapping (first param per depth-bucket):')
    seen_depths = set()
    for name, depth, lr in smoke_diag:
        if depth in seen_depths: continue
        seen_depths.add(depth)
        print(f'    depth={depth:>2d}  lr={lr:.3e}   e.g. `{name}`')
    print(f'    HEAD           lr={FT_HEAD_LR:.3e}')

    print('\n[2] Forward + backward + autocast + GradScaler sanity check...')
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_loader = DataLoader(train_global, batch_size=FT_BATCH, shuffle=True,
                              num_workers=0, collate_fn=collate)
    batch = next(iter(smoke_loader))
    img = batch['image'].to(DEVICE)
    lbl = batch['label'].to(DEVICE)
    weights = compute_class_weights(train_global).to(DEVICE)
    crit  = nn.CrossEntropyLoss(weight=weights)
    opt   = AdamW(smoke_groups)
    scaler_smoke = GradScaler()

    sm.train()
    opt.zero_grad()
    with autocast(dtype=torch.float16):
        logits = sm(img)
        loss = crit(logits, lbl)
    scaler_smoke.scale(loss).backward()
    scaler_smoke.step(opt)
    scaler_smoke.update()
    print(f'  One-step loss: {loss.item():.4f}  (finite: {torch.isfinite(loss).item()})')
    assert torch.isfinite(loss).item(), 'Loss is NaN/Inf on first step -- abort.'

    if torch.cuda.is_available():
        peak_after_step = torch.cuda.max_memory_allocated(DEVICE) / 1e9
        print(f'  Peak GPU after 1 step: {peak_after_step:.2f} GB')
        if peak_after_step > 35.0:
            print('!!' + '=' * 74)
            print(f'!! MEMORY WARNING: peak {peak_after_step:.2f} GB above A100 40 GB comfort.')
            print(f'!! Current MEMORY_FALLBACK_LEVEL = {MEMORY_FALLBACK_LEVEL}. Consider bumping.')
            print('!!' + '=' * 74)
        del sm, sb, opt, scaler_smoke

    print(f'\n[3] Running {SMOKE_EPOCHS}-epoch smoke on seed {SMOKE_SEED} '
          f'(schedule built for {FT_EPOCHS} epochs)...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_result = train_one_seed(
        SMOKE_SEED,
        train_set=train_global,
        val_set=val_global,
        test_set=test_global,
        num_epochs=SMOKE_EPOCHS,
        scheduler_total_epochs=FT_EPOCHS,
        run_name_override=f'{RUN_NAME_PREFIX}_SMOKE_seed{SMOKE_SEED}_finetune',
    )

    hist = smoke_result['history']
    train_losses = [h['train_loss'] for h in hist]
    val_f1s      = [h['val_macro_f1'] for h in hist]
    per_epoch_s  = [h['time_s'] for h in hist]
    per_epoch_lr = [h['current_lr'] for h in hist]

    print('\n' + '=' * 76)
    print('SMOKE SUMMARY')
    print('=' * 76)
    print(f'  train_loss trajectory:   {[f"{l:.4f}" for l in train_losses]}')
    print(f'  val_macro_f1 trajectory: {[f"{f:.4f}" for f in val_f1s]}')
    print(f'  end-of-epoch lr_max:     {[f"{lr:.2e}" for lr in per_epoch_lr]}')
    print(f'  time per epoch:          {[f"{t:.0f}s" for t in per_epoch_s]}')
    print(f'  peak GPU during train:   {smoke_result["peak_gpu_gb"]:.2f} GB')
    print(f'  wall time (smoke seed):  {smoke_result["wall_time_s"]:.0f} s')
    print(f'  grad_ckpt_active:        {smoke_result["grad_ckpt_active"]}')

    avg_epoch_s = float(np.mean(per_epoch_s))
    extrapolated_per_seed_s = avg_epoch_s * FT_EPOCHS
    extrapolated_total_s    = extrapolated_per_seed_s * len(SEEDS)
    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print(f'\n  Extrapolated per-seed:   {_fmt_hms(extrapolated_per_seed_s)}  '
          f'({FT_EPOCHS} epochs x {avg_epoch_s:.0f}s/epoch)')
    print(f'  Extrapolated total:      {_fmt_hms(extrapolated_total_s)}  '
          f'({len(SEEDS)} seeds x {_fmt_hms(extrapolated_per_seed_s)})')

    smoke_out = Path(OUTPUT_DIR) / 'smoke_check_results.json'
    with smoke_out.open('w') as f:
        json.dump(smoke_result, f, indent=2)
    print(f'\nSmoke results written: {smoke_out}')
    print(f'\nSMOKE_ONLY = {SMOKE_ONLY} -- multi-seed cell below will '
          f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


SMOKE_ONLY = False
Smoke check: 2 epochs on seed 314
MEMORY_FALLBACK_LEVEL = 0  (bs=16, accum=1, grad_ckpt=False)
Scheduler built for FULL 25-epoch protocol (previews real LR trajectory).

[1] Instantiate FT model + inspect LLRD depth->LR mapping...
  feature_dim = 768
  Total params:       257,505,357
  Trainable params:   143,519,885   (expect >80,000,000 for full FT)
  LLRD: 12 blocks, max_depth=12, gamma=0.75, base_lr=6e-05
  LLRD: 14 param groups (backbone: 13 x depth-buckets, + 1 head group at lr=0.001)
  LLRD depth->lr: shallowest=1.90e-06 (depth 0), deepest=6.00e-05 (depth 12)

  LLRD depth->LR mapping (first param per depth-bucket):
    depth= 1  lr=2.534e-06   e.g. `backbone.encoder.blocks.0.norm1.weight`
    depth= 2  lr=3.379e-06   e.g. `backbone.encoder.blocks.1.norm1.weight`
    depth= 3  lr=4.505e-06   e.g. `backbone.encoder.blocks.2.norm1.weight`
    depth= 4  lr=6.007e-06   e.g. `backbone.encoder.blocks.3.norm1.weight`
    depth= 5  lr=8.009e-06   e.g. `backbone.encode

/tmp/ipykernel_1448/146655704.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_smoke = GradScaler()
/tmp/ipykernel_1448/146655704.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  One-step loss: 2.7172  (finite: True)
  Peak GPU after 1 step: 9.96 GB

[3] Running 2-epoch smoke on seed 314 (schedule built for 25 epochs)...

--- seed 314  fine-tune ---
    training loop: 2 epochs   scheduler length: 25 epochs
    batch=16  grad_accum=1  effective_batch=16  grad_ckpt=False
  feature_dim = 768
  Total params:       257,505,357
  Trainable params:   143,519,885
  LLRD: 12 blocks, max_depth=12, gamma=0.75, base_lr=6e-05
  LLRD: 14 param groups (backbone: 13 x depth-buckets, + 1 head group at lr=0.001)
  LLRD depth->lr: shallowest=1.90e-06 (depth 0), deepest=6.00e-05 (depth 12)


/tmp/ipykernel_1448/3381767550.py:267: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep   1  loss=2.2870  lr_max=8.02e-04  val_acc=0.2927  val_f1=0.1725 *


/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep   2  loss=1.9385  lr_max=9.98e-04  val_acc=0.3333  val_f1=0.2423 *
  [BEST_CKPT_BEFORE_TEST] restored epoch 2 (val_f1=0.2423) before test

SMOKE SUMMARY
  train_loss trajectory:   ['2.2870', '1.9385']
  val_macro_f1 trajectory: ['0.1725', '0.2423']
  end-of-epoch lr_max:     ['8.02e-04', '9.98e-04']
  time per epoch:          ['85s', '85s']
  peak GPU during train:   10.92 GB
  wall time (smoke seed):  255 s
  grad_ckpt_active:        False

  Extrapolated per-seed:   35m 16s  (25 epochs x 85s/epoch)
  Extrapolated total:      1h 45m 48s  (3 seeds x 35m 16s)

Smoke results written: /content/drive/MyDrive/infra_fm/results/fm_eval_olmoearth_finetune_0.3x_v1/smoke_check_results.json

SMOKE_ONLY = False -- multi-seed cell below will run all 3 seeds.


In [17]:
\
# ============================================================================
# Multi-seed fine-tune + aggregate. Skip-if-JSON-exists guard also cleans up
# stale per-seed checkpoint dirs when the JSON already exists.
# ============================================================================
import json as _json
import numpy as np
import shutil as _shutil

if SMOKE_ONLY:
    print('SMOKE_ONLY=True -- skipping the 3-seed fine-tune run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        out_path   = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        seed_ckpt  = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_finetune'
        if out_path.exists():
            if seed_ckpt.exists():
                try:
                    _shutil.rmtree(seed_ckpt)
                    print(f'  [CLEANUP] deleted stale ckpt dir: {seed_ckpt.name}')
                except OSError as e:
                    print(f'  WARNING: failed to delete stale ckpt dir: {e}')
            with out_path.open() as f:
                per_seed_results[seed] = _json.load(f)['finetune']
            print(f'  [SKIP] seed {seed}: existing per-seed JSON at {out_path.name} '
                  '(delete to force rerun)')
            continue
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        with open(out_path, 'w') as f:
            _json.dump({'finetune': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1':  float(per_class_arr[:, i].mean()),
         'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']
    agg['finetune_protocol']           = FINETUNE_PROTOCOL
    agg['peak_gpu_gb'] = {
        'mean':     float(np.mean([per_seed_results[s]['peak_gpu_gb'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['peak_gpu_gb']) for s in SEEDS},
    }
    agg['wall_time_s'] = {
        'mean':     float(np.mean([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'total':    float(np.sum([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['wall_time_s']) for s in SEEDS},
    }
    agg['llrd_depth_lr_map'] = per_seed_results[SEEDS[-1]]['llrd_depth_lr_map']
    agg['grad_ckpt_active']  = per_seed_results[SEEDS[-1]]['grad_ckpt_active']
    agg['training_subsample_fraction'] = SUBSAMPLE_FRACTION
    agg['training_subsample_seed']     = SUBSAMPLE_SEED
    agg['training_subsample_note'] = (
        'Stratified subsample of the TRAINING set only; val + test '
        'sets unchanged. SUBSAMPLE_SEED=42 is fixed across the 3 '
        'training seeds so the reported std reflects seed variance only, '
        'not subsample variance.'
    )

    _is_full_run = set(SEEDS) == set(FULL_PROTOCOL_SEEDS)
    if _is_full_run:
        agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol '
              f'{FULL_PROTOCOL_SEEDS}. Skipping aggregate JSON write '
              'to preserve any existing 3-seed aggregate.')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap='Oranges', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'OlmoEarth v1.1-Base FT -- aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if _is_full_run:
        cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Confusion matrix saved: {cm_path}')

    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print('\n' + '=' * 76)
    print(f'OlmoEarth v1.1-Base FT -- aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- {agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- {agg["test_accuracy"]["std"]:.4f}')
    print(f'Peak GPU:      {agg["peak_gpu_gb"]["mean"]:.2f} GB (mean across seeds)')
    print(f'Wall clock:    per-seed {_fmt_hms(agg["wall_time_s"]["mean"])}, '
          f'total {_fmt_hms(agg["wall_time_s"]["total"])}')
    print(f'Grad ckpt:     {agg["grad_ckpt_active"]}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')



--- seed 314  fine-tune ---
    training loop: 25 epochs   scheduler length: 25 epochs
    batch=16  grad_accum=1  effective_batch=16  grad_ckpt=False
  feature_dim = 768
  Total params:       257,505,357
  Trainable params:   143,519,885
  LLRD: 12 blocks, max_depth=12, gamma=0.75, base_lr=6e-05
  LLRD: 14 param groups (backbone: 13 x depth-buckets, + 1 head group at lr=0.001)
  LLRD depth->lr: shallowest=1.90e-06 (depth 0), deepest=6.00e-05 (depth 12)


/tmp/ipykernel_1448/3381767550.py:267: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep   1  loss=2.2870  lr_max=8.02e-04  val_acc=0.2927  val_f1=0.1725 *


/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep   2  loss=1.9385  lr_max=9.98e-04  val_acc=0.3337  val_f1=0.2424 *


...

  ep  23  loss=0.0689  lr_max=1.74e-05  val_acc=0.4702  val_f1=0.3419


/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep  24  loss=0.0734  lr_max=4.37e-06  val_acc=0.4699  val_f1=0.3433


/tmp/ipykernel_1448/3381767550.py:321: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  ep  25  loss=0.0707  lr_max=0.00e+00  val_acc=0.4674  val_f1=0.3425
  [BEST_CKPT_BEFORE_TEST] restored epoch 11 (val_f1=0.3506) before test

  saved /content/drive/MyDrive/infra_fm/results/fm_eval_olmoearth_finetune_0.3x_v1/olmoearth_finetune_0.3x_v1_seed161_results.json

Aggregate saved: /content/drive/MyDrive/infra_fm/results/fm_eval_olmoearth_finetune_0.3x_v1/olmoearth_finetune_0.3x_v1_aggregate.json
Confusion matrix saved: /content/drive/MyDrive/infra_fm/results/fm_eval_olmoearth_finetune_0.3x_v1/confusion_matrix_olmoearth_finetune_0.3x_v1_aggregate.png

OlmoEarth v1.1-Base FT -- aggregate (3 seeds)
Test macro F1: 0.3889 +/- 0.0282
Test accuracy: 0.4608 +/- 0.0109
Peak GPU:      10.92 GB (mean across seeds)
Wall clock:    per-seed 39m 43s, total 1h 59m 9s
Grad ckpt:     False

Per-class F1 (mean +/- std):
  [ 0] energy.transmission.substation     0.2050 +/- 0.0185
  [ 1] energy.distribution.substation     0.2428 +/- 0.0127
  [ 2] energy.distribution.other          0.3670 +/- 0.03